Calculate statistical summaries

In [0]:
%sql
--basic statistical summary 
SELECT
    COUNT(*)            AS total_rows,
    COUNT(price)        AS non_null_prices,
    MIN(price)          AS min_price,
    MAX(price)          AS max_price,
    AVG(price)          AS avg_price,
    STDDEV(price)       AS stddev_price
FROM ecommerce.events_delta
WHERE event_type = 'purchase';


total_rows,non_null_prices,min_price,max_price,avg_price,stddev_price
916923,916923,0.77,2574.07,300.12623682686984,341.3823938845062


In [0]:
%sql
--median
SELECT
    PERCENTILE_APPROX(price, 0.5) AS median_price
FROM ecommerce.events_delta
WHERE event_type = 'purchase';


median_price
169.96


In [0]:
%sql
--statistical summary by day 
SELECT
    DATE(event_time) AS event_date,
    COUNT(*) AS orders,
    ROUND(AVG(price), 2) AS avg_price,
    PERCENTILE_APPROX(price, 0.5) AS median_price
FROM ecommerce.events_delta
WHERE event_type = 'purchase'
GROUP BY DATE(event_time)
ORDER BY event_date;


event_date,orders,avg_price,median_price
2019-11-01,22457,309.45,174.73
2019-11-02,21863,292.26,170.13
2019-11-03,22145,300.61,172.08
2019-11-04,26889,298.78,169.96
2019-11-05,24872,291.44,161.65
2019-11-06,25319,292.18,166.1
2019-11-07,24861,284.06,159.94
2019-11-08,25714,301.08,169.92
2019-11-09,22766,291.96,169.75
2019-11-10,22878,289.95,169.28


In [0]:
%sql
--Hypothesis Testing (Weekday vs Weekend) & Hypothesis testing compares two opposite ideas
WITH data AS (
    SELECT
        price,
        CASE
            WHEN dayofweek(event_time) IN (1, 7) THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_type
    FROM ecommerce.events_delta
    WHERE event_type = 'purchase'
)
SELECT *
FROM data
LIMIT 10;


price,day_type
1003.37,Weekday
218.51,Weekend
898.35,Weekend
290.42,Weekday
1672.86,Weekday
797.7,Weekend
90.06,Weekday
40.28,Weekday
46.31,Weekend
154.16,Weekend


In [0]:
%sql
--Compute Statistical Summaries
WITH base_events AS (
    SELECT
        price,
        CASE
            WHEN dayofweek(event_time) IN (1, 7) THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_type
    FROM ecommerce.events_delta
    WHERE event_type = 'purchase'
),

stats AS (
    SELECT
        day_type,
        COUNT(*) AS count,
        AVG(price) AS avg_price,
        STDDEV(price) AS stddev_price
    FROM base_events
    GROUP BY day_type
)

SELECT *
FROM stats;


day_type,count,avg_price,stddev_price
Weekday,500250,293.32120783608576,340.4801905598857
Weekend,416673,308.29623044929536,342.2841583017207


In [0]:
%sql
WITH base_events AS (
    SELECT
        price,
        CASE
            WHEN dayofweek(event_time) IN (1, 7) THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_type
    FROM ecommerce.events_delta
    WHERE event_type = 'purchase'
),

stats AS (
    SELECT
        day_type,
        COUNT(*) AS n,
        AVG(price) AS avg_price,
        STDDEV(price) AS stddev_price
    FROM base_events
    GROUP BY day_type
),

weekday_stats AS (
    SELECT
        avg_price  AS weekday_avg,
        stddev_price AS weekday_std,
        n AS weekday_n
    FROM stats
    WHERE day_type = 'Weekday'
),

weekend_stats AS (
    SELECT
        avg_price  AS weekend_avg,
        stddev_price AS weekend_std,
        n AS weekend_n
    FROM stats
    WHERE day_type = 'Weekend'
),

combined_stats AS (
    SELECT
        w.weekday_avg,
        e.weekend_avg,
        w.weekday_std,
        e.weekend_std,
        w.weekday_n,
        e.weekend_n
    FROM weekday_stats w
    CROSS JOIN weekend_stats e
),

t_test AS (
    SELECT
        weekday_avg,
        weekend_avg,
        (weekday_avg - weekend_avg)
        /
        SQRT(
            (POWER(weekday_std, 2) / weekday_n)
          + (POWER(weekend_std, 2) / weekend_n)
        ) AS t_statistic
    FROM combined_stats
)

SELECT *
FROM t_test;


weekday_avg,weekend_avg,t_statistic
293.32120783608576,308.29623044929536,-20.909582444296134


In [0]:
%sql
--Identify correlations
WITH daily_metrics AS (
    SELECT
        DATE(event_time) AS event_date,
        COUNT(*) AS orders,
        SUM(price) AS revenue
    FROM ecommerce.events_delta
    WHERE event_type = 'purchase'
    GROUP BY DATE(event_time)
)

SELECT
    corr(orders, revenue) AS orders_revenue_corr
FROM daily_metrics;



orders_revenue_corr
0.9986963194890345
